# NeuroHertz v2: full run on Colab

Unzip `neurohertz_v2.zip` anywhere in your Google Drive (for example `MyDrive/neurohertz_v2`). The data
folders stay where they are (`MyDrive/data`, `MyDrive/ds004504-download*`, ...). Results are written
inside the `neurohertz_v2/results` folder on Drive, so they survive the session ending.

Run cells top to bottom. Cells marked **LABELS** open diagnoses; everything before them is label-blind.
Read `docs/LEAKAGE_CONTRACT.md` and `docs/MECHANISMS.md` first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# where you unzipped the code, and the Drive root that holds the data folders
NH_ROOT  = '/content/drive/MyDrive/neurohertz_v2'
os.environ['NH_DRIVE'] = '/content/drive/MyDrive'
os.chdir(NH_ROOT)
JOBS = os.cpu_count()
print('code:', NH_ROOT, '| data root:', os.environ['NH_DRIVE'], '| cores:', JOBS)

In [ ]:
!pip -q install -r requirements.txt
!python -m pytest -q tests   # leakage guards + method checks: everything must pass before going on

## 1. Where is the data? (label-blind)
If a dataset shows 0 folders, edit `config/datasets.yaml` or pass `--roots` to the scripts.

In [ ]:
import sys; sys.path.insert(0, NH_ROOT)
from nh.datasets import get_adapter
for name in ['ds004504', 'ds006036']:
    ad = get_adapter(name)
    print(name, '->', ad.roots or 'NOT FOUND')
    if ad.roots:
        inv = ad.find_recordings(); print('   files:', len(inv), '| subjects:', inv.subject.nunique())

## 2. Stage A: label-blind features
~2-3 min per subject per core with 20 surrogates. Re-running resumes from `per_subject/`.

In [ ]:
!python scripts/01_extract_features.py --dataset ds004504 --jobs {JOBS} --split-half

In [ ]:
# optional: eyes-open with flicker (flicker-free stretches only); needed for test M17
!python scripts/01_extract_features.py --dataset ds006036 --jobs {JOBS}

## 3. Stage 0: simulation calibration and primary-metric choice (label-free)

In [ ]:
!python scripts/02b_calibrate_simulation.py --dataset ds004504
!python scripts/02_simulation_validation.py --seeds 8 --n-surr 10 --jobs {JOBS}

## 4. Method ablations, chosen without labels (optional but recommended; several hours)

In [ ]:
!python scripts/10_run_ablations.py --mode simulation --jobs {JOBS}
!python scripts/10_run_ablations.py --mode one-at-a-time --dataset ds004504 --jobs {JOBS}
!python scripts/11_select_methods.py --dataset ds004504
# then copy the chosen options from registry/method_selection.json into config/pipeline.yaml
# !python scripts/10_run_ablations.py --mode factorial --dataset ds004504 --jobs {JOBS}   # 36 pipelines, for the spec curve

## 5. Mechanism fingerprints (label-free) and a synthetic positive control

The synthetic run injects a KNOWN mechanism into fake EEG with our group sizes. If step 14 does not
point to it, do not trust the mechanistic analysis on real data.

In [ ]:
!python scripts/13_mechanism_fingerprints.py --jobs {JOBS}

In [ ]:
!python scripts/00_make_synthetic_dataset.py --mechanism synapses_onto_pv --k 0.75 --n 29 36 23
!python scripts/01_extract_features.py --dataset synthetic --no-ica --jobs {JOBS}
!python scripts/14_mechanism_tests.py --dataset synthetic
# second control, other side of the split:
# !rm -rf results/synthetic_dataset results/features/synthetic
# !python scripts/00_make_synthetic_dataset.py --mechanism erratic_drive --k 0.75 --n 29 36 23  (then 01 and 14 again)

## 6. Freeze

Look only at `qc.csv`, the simulation outputs and the pooled label-free descriptives. Then set
`status: frozen` and `statistics.primary_feature` in `config/pipeline.yaml` and run the freeze.
After this, changing any config or code makes the gate refuse confirmatory label access.

In [ ]:
!python scripts/03_freeze.py

## 7. LABELS: discovery analyses (logged in `registry/access_ledger.jsonl`)

Before the freeze these scripts refuse to open labels. For development only, add `--dev-unblind-reason "..."`: the opening is logged and every output is stamped EXPLORATORY.

In [ ]:
!python scripts/04_discovery_analysis.py --dataset ds004504
!python scripts/05_prediction.py --dataset ds004504
!python scripts/14_mechanism_tests.py --dataset ds004504
!python scripts/12_specification_curve.py --dataset ds004504
!python scripts/09_ceiling_check.py --dataset ds004504

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('results/analysis/ds004504/primary/mechanism/report.md').read()))